# Part 3 · Challenge: event day from Centraal Station to Museumplein

**The network is the real street map.** Every walkable street, square and footpath within about 650 m of
the line from Centraal Station to Museumplein comes straight from **OpenStreetMap**: about 350 junctions
and 650 street segments, with their real lengths. The numbered dots are the main places:

| node | place | | node | place |
|---|---|---|---|---|
| **0** | **Centraal Station** (start) | | 5 | Leidseplein |
| 1 | Stationsplein | | 6 | Rembrandtplein |
| 2 | Dam (Madame Tussauds) | | 7 | Weteringcircuit |
| 3 | Spui | | 8 | Rijksmuseum |
| 4 | Muntplein | | **9** | **Museumplein** (the event) |

Grey streets are empty; coloured streets have people on them (green = free walking, red = jammed).
A big open-air event starts on **Museumplein** and the trains bring thousands of visitors into
**Amsterdam Centraal**. Stewards on the platforms decide **how many people per minute** are let out of
the station towards the event. That is your first lever: **you design the arrival pattern** for the first hour.
Your second lever: stewards on **Stationsplein** can **close routes**, block by block.

The obvious way out is **Damrak**, past Madame Tussauds on the Dam: a *very* wide street (20 m),
so it looks like it can take anyone. But south of the Dam the crowd has to fit into much narrower
streets: **Rokin**, then **Vijzelstraat** towards the Museum Quarter. Other routes exist (Kalverstraat and
Leidsestraat, Nieuwezijds Voorburgwal and the canals), but they are longer. People choose their own
route (shorter, wider and emptier streets are more attractive), just like in Part 1, among **5 routes**.
All 5 start at Stationsplein, so stewards there can send people away from a route by closing its entrance.

### Rules
* You set the number of people per minute leaving the station for six 10-minute blocks (minute 0 to 60).
* For each of the 5 routes you decide, per 10-minute block, whether it is **open** or **closed**. Closing a
  route shuts the gate at its first street (in PedNStream: gate width 0), so nobody new can walk in; people
  already on it keep walking, and people in the station pick another open route. After minute 60 the last
  setting stays. At least one route must be open in every block.
* The simulation runs for **2 hours**. The shortest walk is about **3.3 km**, ≈ 41 minutes on empty streets;
  because people spread over several routes, the average trip on empty streets is about **50 minutes**.
* **Score = people who reach Museumplein − hours that people spend on the streets (person-hours).**
  Every visitor you deliver earns 1 point; every hour someone spends walking or waiting costs 1 point.
  So a visitor whose trip takes longer than an hour costs more than they bring in. With a 50-minute walk
  there is not much room for queueing! **Higher is better.**
* Post your best score on the leaderboard.

**How to use this notebook**
* Press **▶** on the left of a grey cell (or click it and press **Shift + Enter**) to run it.
* Cells with sliders and boxes are *forms*: change the values on the right, then press ▶ again.
* You never have to edit code. If you're curious, double-click a form to see the code behind it.
* Something broken? **Runtime → Restart session**, then run the first cell again.

## ▶ Step 1: install (run this first, ≈30 s)
This is how you install **any** Python package from GitHub. `pednstream-workshop` is a small
package with this workshop's network and plotting helpers; installing it also installs
**PedNStream** itself. If Colab restarted, run it again.
You can ignore the note *"you may need to restart the kernel"*.

In [ ]:
%pip install -q "pednstream-workshop @ git+https://github.com/WaimenMak/pednstream-workshop.git"
import json, importlib.metadata as md
if json.loads(md.distribution("pednstream").read_text("direct_url.json") or "{}").get("vcs_info", {}).get("commit_id") != "4a85236b3c47b17f25c2bfe8d5c0ef35651900a7":
    %pip install -q --force-reinstall --no-deps "pednstream @ git+https://github.com/WaimenMak/PedNStream.git@4a85236b3c47b17f25c2bfe8d5c0ef35651900a7"

import matplotlib.pyplot as plt
import pednstream_workshop
if tuple(map(int, pednstream_workshop.__version__.split("."))) < (1, 7, 0):
    raise RuntimeError("An old version of the workshop package is loaded. "
                       "Click Runtime → Restart session, then run this cell again.")
from pednstream_workshop import run, AMSTERDAM, arrival_plan, route_plan, show_routes
plt.rcParams["figure.dpi"] = 90
print("✅ Ready! Scroll down and continue.")

## ▶ Step 2: load the Amsterdam tools and look at the 5 routes
Routes that share a street run side by side on this map. The route numbers are the ones you open and close below.

In [ ]:
import pandas as pd
from pednstream_workshop import AMSTERDAM, arrival_plan, route_plan, routes, show_routes
show_routes(AMSTERDAM); plt.show()
display(routes(AMSTERDAM).drop(columns=["OD", "nodes"]))

In [ ]:
#@title 🎟️ **Your arrival plan**: set the values, then press ▶ {display-mode: "form"}
team_name = "my team"  #@param {type:"string"}
#@markdown People per minute let out of Centraal Station, for each 10-minute block of the first hour
minutes_00_10 = 300  #@param {type:"slider", min:0, max:900, step:30}
minutes_10_20 = 300  #@param {type:"slider", min:0, max:900, step:30}
minutes_20_30 = 300  #@param {type:"slider", min:0, max:900, step:30}
minutes_30_40 = 300  #@param {type:"slider", min:0, max:900, step:30}
minutes_40_50 = 300  #@param {type:"slider", min:0, max:900, step:30}
minutes_50_60 = 300  #@param {type:"slider", min:0, max:900, step:30}
#@markdown **Routes open per 10-minute block**: six digits, one per block (1 = open, 0 = closed).
#@markdown E.g. "000111" = closed in minutes 0-30, open from minute 30.
route_1_Damrak = "111111"  #@param {type:"string"}
route_2_Kalverstraat = "111111"  #@param {type:"string"}
route_3_Nieuwezijds_Voorburgwal = "111111"  #@param {type:"string"}
route_4_Oudezijds_Achterburgwal = "111111"  #@param {type:"string"}
route_5_Singel = "111111"  #@param {type:"string"}

plan = [minutes_00_10, minutes_10_20, minutes_20_30, minutes_30_40, minutes_40_50, minutes_50_60]
route_codes = [route_1_Damrak, route_2_Kalverstraat, route_3_Nieuwezijds_Voorburgwal,
               route_4_Oudezijds_Achterburgwal, route_5_Singel]
for i, code_ in enumerate(route_codes, start=1):
    if len(code_.replace(" ", "")) != 6 or set(code_.replace(" ", "")) - {"0", "1"}:
        raise ValueError(f"Route {i}: use six digits of 0 and 1, e.g. '111111' (you typed {code_!r})")
res = run(network_dir=AMSTERDAM, arrivals={0: arrival_plan(plan)}, open_routes=route_plan(*route_codes))
last_plan = res   # remembered for the animation below
k = res.kpis()
sent, arrived = k["total demand (people)"], k["reached Museumplein"]
hours = float(k["time spent inside (person-hours)"])
score = arrived - hours

if "plans" not in globals():
    plans = []
plans.append({"team": team_name, **{f"{10 * i}-{10 * i + 10}'": p for i, p in enumerate(plan)},
              "routes 1-5": " ".join(route_codes), "sent": sent, "arrived": arrived,
              "person-hours": round(hours), "score": round(score)})

print(f"🚆 Sent {sent:,} people   🎨 {arrived:,} reached Museumplein   ⏱ {hours:,.0f} person-hours on the streets")
print(f"   average trip: {60 * hours / max(sent, 1):.0f} min (empty streets: about 50 min)")
print(f"\n   SCORE = {score:,.0f}\n")
print(f"📋 Leaderboard line:  {team_name} | plan {plan} | routes {' '.join(route_codes)} | score {score:.0f}")
res.dashboard(f"{team_name}: {plan} people/min", flow=True); plt.show()
print("Your plans so far (best first):")
display(pd.DataFrame(plans).sort_values("score", ascending=False).reset_index(drop=True))

### 🎬 Watch your plan play out
Run your plan above first, then this cell. It replays the **last plan you ran**: the map shows how crowded
each street is, minute by minute, and the black line on the right shows where you are in time.
Press ▶ under the animation; use − / + to slow down or speed up. (Takes about a minute to draw.)

In [ ]:
#@title 🎬 Animate your last plan {display-mode: "form"}
#@markdown Show one frame every ... simulated minutes (smaller = smoother but slower to draw)
minutes_per_frame = 2  #@param {type:"slider", min:0.5, max:5, step:0.5}
if "last_plan" not in globals():
    print("⚠️ Run your arrival plan (the 🎟️ cell above) first.")
else:
    display(last_plan.animate(every=max(1, round(minutes_per_frame * 6)), flow=True))

### Stuck? Hints (open one at a time)
<details><summary>Hint 1</summary>Look at the <i>In and out</i> plot. Does the red line (arrivals per minute)
follow your blue blocks, about 50 minutes later? When does it stop following them?</details>
<details><summary>Hint 2</summary>When too many people arrive at once, they wait somewhere. Where? Watch the
animation: which street fills up first, and where does the queue go next? Is it the wide Damrak?</details>
<details><summary>Hint 3</summary>Someone who leaves at minute 55 needs about 50 minutes on empty streets.
Do they make it before the end (minute 120)? What if they have to queue?</details>
<details><summary>Hint 4</summary>Your blocks don't have to be equal. A bigger group is fine as long as
the queue it causes clears quickly. Is there a moment in the hour when nobody comes after them?</details>
<details><summary>Hint 5</summary>Look at the route map: which routes end up in the same narrow street? Almost
everyone picks the route past the wide Damrak. What happens further on if fewer people start that way?</details>

---
### For coders (optional): let the computer search
A simple search in two steps. First try a few constant arrival rates, each with all routes open and with
one route closed for the whole hour. Then take the best and move each block up or down by 60, keeping any
change that improves the score. Each run takes about 10 s on the full street map, so this takes **≈5 minutes**.
Can you beat it by hand, for example by closing routes only in some blocks? Can you write a better search?

In [ ]:
def amsterdam_score(plan, open_routes=None):
    k = run(network_dir=AMSTERDAM, arrivals={0: arrival_plan(plan)}, open_routes=open_routes).kpis()
    return k["reached Museumplein"] - float(k["time spent inside (person-hours)"])

options = {"all open": [True] * 5}
for r in range(5):
    options[f"route {r + 1} closed"] = [i != r for i in range(5)]

trials = {(rate, name): amsterdam_score([rate] * 6, open_) for rate in [360, 480, 600] for name, open_ in options.items()}
(rate, name), best_score = max(trials.items(), key=lambda kv: kv[1])
best, open_routes = [rate] * 6, options[name]
print(f"step 1: {rate}/min with {name}, score {best_score:,.0f}")

for i in range(6):
    for change in (-60, 60):
        trial = best.copy()
        trial[i] = min(900, max(0, trial[i] + change))
        s = amsterdam_score(trial, open_routes)
        if s > best_score + 1:
            best, best_score = trial, s
print(f"step 2: plan {best} with {name}, score {best_score:,.0f}")

### Reflection
1. Describe your arrival plan in words a steward on the platform would understand.
2. Why does it work? Name the street that decides how many people per minute can reach Museumplein.
3. Damrak is 20 m wide. Did that help or hurt? What does a very wide street do to *where* people choose to walk?
4. Most people walk Damrak, Rokin and Vijzelstraat even when Vijzelstraat is packed and Kalverstraat is quiet.
   In PedNStream people choose at each junction from the distance and how wide and empty the *next* street is;
   they can't see a jam 2 km ahead. Would real visitors (with phones, or stewards) behave the same way?
5. Which routes did you close, and when? Closing a route takes a choice away from people. Why can that make
   the trip faster for everyone? Would you close the same routes in real life?